In [1]:
import numpy as np
import matplotlib.pyplot as plt

<h3>Implement convolution forward propagation</h3>

In [2]:
def initialize_params(f, num_channels, filters):
    
    W = np.random.randn(filters, f, f, num_channels) * np.sqrt(2 / f)
    b = np.random.randn(filters, 1, 1, 1) * np.sqrt(2 / f)

    return (W, b)

In [3]:
def zero_padding(X, padding):
  
    X_pad = np.pad(X, ((0, 0), (padding, padding), (padding, padding), (0, 0)), mode="constant", constant_values=(0, 0))    
    
    return X_pad

In [4]:
n = np.ones((1, 3, 3, 1))
o = zero_padding(n, 3)
print(o.shape)
print(n.shape)

(1, 9, 9, 1)
(1, 3, 3, 1)


In [5]:
def x_conv_shape(n_pre, f, s):
    
    n = int((n_pre - f / s)) + 1
    
    return n

In [6]:
n = 10
f = 3
p = 2
s =1
nn = x_conv_shape(n, f, s)
nn

8

In [7]:
def relu(Z):
    
    A = np.maximum(0, Z)
    cache = Z
    
    return A, cache

In [8]:
def conv_1_filter(X, W, nh, nw, stride):
    
    (f, f, n_c) = W.shape
    X_conv = np.zeros((nh, nw))

    for k in range(n_c):

        for i in range(nh):
            v_start = i * stride
            v_end =  v_start + f
            for j in range(nw):
                h_start = j * stride
                h_end =  h_start + f
                
                frame = X[v_start:v_end, h_start:h_end, k]
                X_conv[i, j] += np.sum(frame * W[:, :, k])        
            
    return X_conv

In [9]:
x = np.ones((10, 10, 2))
x_pad = np.ones((14, 14, 2))
w = np.random.randn(3, 3, 2)
x_conv = conv_1_filter(x,  w, 3, 3, 3)
r,c = relu(x_conv)
print(r.shape)
print(c.shape)
print(x.shape)
print(w.shape)
print(x_conv.shape)

(3, 3)
(3, 3)
(10, 10, 2)
(3, 3, 2)
(3, 3)


In [10]:
def conv2d(X, filters, kernel_size, padding, strides, parameters=None, layer=None):
    """
    Applies convolutional process
    Args:
        - X(ndarray): input data with the shape of (m, n_h, n_W, n_C)
        - filters(int): number of applied filters
        - kernel_size(tuple): size of the applied filters (f, f)
        - padding(int): number of padding columns or rows
        - strides(int): number of strides
        - parameters(dictionary): contains W and b matrices
        - layer(int): layer number
    """

    num_channels = X[0].shape[-1]
    f = kernel_size[0]
    

    if not parameters:
        (W, b) = initialize_params(f=f, num_channels=num_channels, filters=filters)
    else:
        (W, b) = parameters[f"W{layer}"], parameters[f"b{layer}"]
      
    linear_cache = (X, W, b, filters, kernel_size, padding, strides, layer)
    X_pad = zero_padding(X=X, padding=padding)
    
    nh = x_conv_shape(n_pre=X_pad[0].shape[0], f=f, s=strides)
    nw = x_conv_shape(n_pre=X_pad[0].shape[1], f=f, s=strides)
   
    m = X_pad.shape[0]

    X_conv = np.zeros((m, nh, nw, filters))
    activation_cache = np.zeros((m, nh, nw, filters))
    
    for i in range(m):
        X_temp = np.zeros((nh, nw, filters))
       
        for f in range(filters):
            
            Z = conv_1_filter(X=X_pad[i, :, :, :], W=W[f, :, :, :], nh=nh, nw=nw, stride=strides) + b[f]
            X_temp[:, :, f], activation_cache[i, :, :, f] = relu(Z=Z)
   
        X_conv[i, :, :, :] = X_temp

    caches = (linear_cache, activation_cache)

    return X_conv, caches      

In [11]:
x = np.ones((12, 6, 6, 2))
w = np.random.randn(12, 3, 3, 2)
b = np.random.randn(12, 1, 1, 1)
x_conv1, c1 = conv2d(X=x, filters=4, kernel_size=(3, 3), padding=2, strides=1, parameters=None, layer=None)
x_conv2, c2 = conv2d(X=x, filters=4, kernel_size=(3, 3), padding=2, strides=1, parameters={"W1": w, "b1": b}, layer=1)
print(x_conv1.shape)
print(x_conv2.shape)

(12, 8, 8, 4)
(12, 8, 8, 4)


<h3>Implements convolution backward propagation</h3>

In [12]:
def zero_padding(X, padding):
  
    X_pad = np.pad(X, ((0, 0), (padding, padding), (padding, padding), (0, 0)), mode="constant", constant_values=(0, 0))    
    
    return X_pad

In [13]:
def backward_relu(dA, Z):
    
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
      
    return dZ

In [14]:
da = np.ones((3, 2, 2, 2))
z = np.random.randn(3, 2,2,2)
r = backward_relu(da, z)
print(da.shape)
print(z.shape)
print(r.shape)
print(r)

(3, 2, 2, 2)
(3, 2, 2, 2)
(3, 2, 2, 2)
[[[[0. 1.]
   [1. 1.]]

  [[1. 0.]
   [1. 1.]]]


 [[[1. 1.]
   [0. 0.]]

  [[1. 0.]
   [1. 0.]]]


 [[[1. 1.]
   [0. 1.]]

  [[0. 1.]
   [1. 0.]]]]


In [15]:
def conv_1_backward(Al, dAl_1, W, dZ, dW, db, step, nh, nw, nc, stride, f):

    for h in range(nh):
            for w in range(nw):               
                for c in range(nc):
                    
                    v_start = h * stride
                    v_end = v_start + f
                    h_start = w * stride
                    h_end = h_start + f
                    
                    a_slice = Al[v_start:v_end, h_start:h_end, :]
                    
                    dAl_1[v_start:v_end, h_start:h_end, :] += W[c, :, :, :] * dZ[step, h, w, c]
                    dW[c, :, :, :] += a_slice * dZ[step, h, w, c]
                    db[c, :, :, :] += dZ[step, h, w, c]
                    
    return (dAl_1, dW, db)    

In [16]:
def conv2d_backward(dAl, caches):

    (linear_cache, activation_cache) = caches
    # linear_cache stored Z for each filter (conv_1_filter), which was stored with relu function
    # linear_cache = (X, W, b, filters, kernel_size, padding, strides, layer)
    (X, W, b, filters, kernel_size, padding, strides, layer) = linear_cache
    # filters (int): Number of applied filters
    # kernel_size (tuple): Size of the applied filters (f, f)
    # padding (int): Number of padding columns or rows
    # strides (int): Number of strides
    # layer (int): Layer number
    
    Z = activation_cache # a 4d matrix (m, nh, nw, filters)

    dZ = backward_relu(dA=dAl, Z=Z)

    (m, nh_pre, nw_pre, nc_pre) = X.shape
    (nc, f, f, nc_pre) = W.shape
    (m, nh, nw, nc) = dZ.shape
    
    dAl_1 = np.zeros(X.shape)                          
    dW = np.zeros(W.shape)
    db = np.zeros(b.shape)
    
    # Pad X and dAl_1
    X_pad = zero_padding(X=X, padding=padding)
    dAl_1_pad = zero_padding(X=dAl_1, padding=padding)
    
    for i in range(m):
        
        x_pad = X_pad[i]
        dal_1_pad = dAl_1_pad[i]

        (dal_1_pad, dW, db) = conv_1_backward(
            Al=x_pad, 
            dAl_1=dal_1_pad, 
            W=W, 
            dZ=dZ, 
            dW=dW, 
            db=db, 
            step=i,
            nh=nh,
            nw=nw,
            nc=nc,
            stride=strides, 
            f=f
        )
    # Remove padding from dAl_1_pad   
    dAl_1 = dAl_1_pad[:, padding:-padding, padding:-padding, :]
    assert(dAl_1.shape == (m, nh_pre, nw_pre, nc_pre))
    
    return dAl_1, dW, db   

In [17]:
dal, dw, db = conv2d_backward(x_conv2, c2)
print(dal.shape)
print(dw.shape)
print(db.shape)

(12, 6, 6, 2)
(12, 3, 3, 2)
(12, 1, 1, 1)


<h3>Implement max and average pooling forward</h3>

In [18]:
def compute_output_size(nh_pre, nw_pre, nc_pre, f, stride):

    nh = int(1 + (nh_pre - f) / stride)
    nw = int(1 + (nw_pre - f) / stride)
    nc = nc_pre
    
    return (nh, nw, nc)

In [19]:
def pool_max_forward(A, f, stride):

    (m, nh_pre, nw_pre, nc_pre) = A.shape
    (nh, nw, nc) = compute_output_size(nh_pre=nh_pre, nw_pre=nw_pre, nc_pre=nc_pre, f=f, stride=stride)
    
    A_pool = np.zeros((m, nh, nw, nc))

    for i in range(m):
        
        for h in range(nh):
            
            v_start = h * stride
            v_end = v_start + f
            
            for w in range(nw):
                
                h_start = w * stride
                h_end = h_start + f

                for c in range(nc):
                    
                    a_slice = A[i, v_start:v_end, h_start:h_end, c]
                    A_pool[i, h, w, c] = np.max(a_slice, axis=(0, 1))

    cache = (A, f, stride)

    return (A_pool, cache)  

In [20]:
a, cache = pool_max_forward(A=x_conv1, f=2, stride=2)
print(a.shape)

(12, 4, 4, 4)


In [21]:
def pool_average_forward(A, f, stride):

    (m, nh_pre, nw_pre, nc_pre) = A.shape
    (nh, nw, nc) = compute_output_size(nh_pre=nh_pre, nw_pre=nw_pre, nc_pre=nc_pre, f=f, stride=stride)
    
    A_pool = np.zeros((m, nh, nw, nc))

    for i in range(m):
        
        for h in range(nh):
            
            v_start = h * stride
            v_end = v_start + f
            
            for w in range(nw):
                
                h_start = w * stride
                h_end = h_start + f

                for c in range(nc):
                    
                    a_slice = A[i, v_start:v_end, h_start:h_end, c]
                    A_pool[i, h, w, c] = np.mean(a_slice, axis=(0, 1))

    cache = (A, f, stride)

    return (A_pool, cache)  

In [22]:
a, cache = pool_average_forward(A=x_conv1, f=2, stride=3)
print(a.shape)

(12, 3, 3, 4)


<h3>Implement max and average pooling backward</h3>

In [23]:
def distribute_value(dZ, shape):
    
    (nh, nw) = shape
    average = dZ / (nh * nw)
    A = np.ones(shape) * average
    
    return A

In [24]:
def pool_max_backward(dA, cache):
 
    (A , f, stride) = cache

   
    m, nh_pre, nw_pre, nc_pre = A.shape
    m, nh, nw, nc = dA.shape

    dA_pre = np.zeros(A.shape)

    
    for i in range(m):
    
        a_pre = A[i]

        for h in range(nh): 
            for w in range(nw):  
                for c in range(nc):  

                    v_start = h * stride
                    v_end = v_start + f
                    h_start = w * stride
                    h_end = h_start + f
                 
                    a_slice = a_pre[v_start: v_end, h_start: h_end, c]
                    mask = (a_slice == np.max(a_slice))
                    dA_pre[i, v_start: v_end, h_start: h_end, c] += mask * dA[i, h, w, c]
                    
    
                    
    return dA_pre

In [25]:
da = pool_max_backward(a, cache)
da.shape

(12, 8, 8, 4)

In [26]:
def pool_average_backward(dA, cache):
 
    (A , f, stride) = cache

    m, nh, nw, nc = dA.shape

    dA_pre = np.zeros(A.shape)

    
    for i in range(m):
    

        for h in range(nh): 
            for w in range(nw):  
                for c in range(nc):  

                    v_start = h * stride
                    v_end = v_start + f
                    h_start = w * stride
                    h_end = h_start + f
                   
                    da = dA[i, h, w, c]                        
                    shape = (f, f)
                    dA_pre[i, v_start: v_end, h_start: h_end, c] += distribute_value(dZ=da, shape=shape)
    
                    
    return dA_pre

In [27]:
da = pool_average_backward(a, cache)
da.shape

(12, 8, 8, 4)

<h3>Implement Fully Connected Forward</h3>

In [28]:
def initialize_parameters(A, units, seed):

    np.random.seed(seed)

    W = np.random.randn(units, len(A)) * np.sqrt(2 / len(A))
    b = np.zeros((units, 1))
    
    return (W, b)

In [29]:
def relu(Z):
    
    A = np.maximum(0, Z)
    cache = Z

    return (A, cache)

In [30]:
def sigmoid(Z):
    
    A = 1 / (1 + np.exp(-Z))
    cache = Z

    return (A, cache)

In [31]:
def softmax(Z):

    A = np.divide(np.exp(Z), np.sum(np.exp(Z), axis=0, keepdims=True) + 1e-15)

    cache = Z

    return (A, cache)

In [32]:
def fc_forward(A, units, activation="relu", W=None, b=None, seed=0):

    if W is None and b is None:
        
        (W, b) = initialize_parameters(
            A=A,
            units=units,
            seed=seed
        )
        
    linear_cache = (A, W, b)
        
    Z = np.dot(W, A) + b

    if activation == "relu":
        
        (A, activation_cache) = relu(Z=Z)
        
    elif activation == "sigmoid":

        (A, activation_cache) = sigmoid(Z=Z)

    elif activation == "softmax":

        (A, activation_cache) = softmax(Z=Z)

    cache = (linear_cache, activation_cache)

    return (A, W, b, cache)   

In [33]:
af = np.random.randn(512)

(a, w, b, cache) = fc_forward(af, 10, activation="relu")
print(a.shape)
print(w.shape)
print(b.shape)

(10, 10)
(10, 512)
(10, 1)


In [34]:
af = np.random.randn(512)

(a, w, b, cache) = fc_forward(af, 10, activation="sigmoid")
print(a.shape)
print(w.shape)
print(b.shape)

(10, 10)
(10, 512)
(10, 1)


In [35]:
af = np.random.randn(512)

(a, w, b, cache) = fc_forward(af, 10, activation="softmax")
print(a.shape)
print(w.shape)
print(b.shape)

(10, 10)
(10, 512)
(10, 1)


In [42]:
af = np.random.randn(512)
w = np.random.rand(10, 512)
b = np.random.randn(10, 1)

(a, w, b, cache) = fc_forward(af, 10, activation="softmax", W=w, b=b)
print(a.shape)
print(w.shape)
print(b.shape)

(10, 10)
(10, 512)
(10, 1)


<h3>Implement Fully Connected Backward</h3>

In [37]:
def backward_relu(dA, Z):
    
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
      
    return dZ

In [38]:
def backward_sigmoid(dA, Z):
    
    A = 1 / (1 + np.exp(-Z))
    dZ = np.multiply(np.multiply(dA, A), 1 - A)

    return dZ

In [39]:
def backward_softmax(dA, Z):
    
    A = np.divide(np.exp(Z), np.sum(np.exp(Z), axis=0, keepdims=True) + 1e-15)
    dZ = A * (1 - A) * dA
    
    return dZ

In [47]:
def fc_backward(A, Y, cache, activation, dA=None, last_layer=False):

    if last_layer:
        dA = - (np.divide(Y, A + 1e-15) - np.divide(1 - Y, 1 - A + 1e-15))
        
    (linear_cache, activation_cache) = cache
    (A, W, b) = linear_cache
    Z = activation_cache
    m = A.shape[1]

    if activation == "sigmoid":
        dZ = backward_sigmoid(dA=dA, Z=Z)

    elif activation == "softmax":
        dZ = backward_softmax(dA=dA, Z=Z)

    elif activation == "relu":
        dZ = backward_relu(dA=dA, Z=Z)

    dA = np.dot(W.T, dZ)
    dW = 1 / m * np.dot(dZ, A.T)
    db = 1 / m * np.sum(dZ, axis=1, keepdims=True)

    return (dA, dW, db)

In [50]:
A = np.array([[0.8, 0.4], [0.2, 0.9]])
Y = np.array([[1, 0], [0, 1]])
A_prev = np.array([[0.5, 0.2], [0.3, 0.4], [0.7, 0.5]])
W = np.array([[0.2, 0.3, 0.5], [0.6, 0.7, 0.9]])
b = np.array([[0.1], [0.2]])
Z = np.dot(W, A_prev) + b
cache = ((A_prev, W, b), Z)
dA = np.array([[0.1, -0.2], [-0.4, 0.3]])

dA_prev, dW, db = fc_backward(A, Y, cache, "sigmoid", dA)
print("Test Case 1 - Sigmoid activation, not last layer")
print("dA_prev:", dA_prev)
print("dW:", dW)
print("db:", db)
print()

# Test case 2: ReLU activation, not last layer
dA = np.array([[0.2, -0.1], [0.4, 0.5]])
dA_prev, dW, db = fc_backward(A, Y, cache, "relu", dA)
print("Test Case 2 - ReLU activation, not last layer")
print("dA_prev:", dA_prev)
print("dW:", dW)
print("db:", db)
print()

# Test case 3: Softmax activation, last layer
A = np.array([[0.8, 0.4], [0.2, 0.9]])
Y = np.array([[1, 0], [0, 1]])
cache = ((A_prev, W, b), Z)

dA_prev, dW, db = fc_backward(A, Y, cache, "softmax", last_layer=True)
print("Test Case 3 - Softmax activation, last layer")
print("dA_prev:", dA_prev)
print("dW:", dW)
print("db:", db)
print()

Test Case 1 - Sigmoid activation, not last layer
dA_prev: [[-0.03494688  0.02518801]
 [-0.03926436  0.02626036]
 [-0.0478993   0.02840508]]
dW: [[ 0.00096282 -0.00598616 -0.00380937]
 [-0.01068414  0.00165467 -0.00862089]]
db: [[-0.01213977]
 [-0.00408581]]

Test Case 2 - ReLU activation, not last layer
dA_prev: [[0.28 0.28]
 [0.34 0.32]
 [0.46 0.4 ]]
dW: [[0.04  0.01  0.045]
 [0.15  0.16  0.265]]
db: [[0.05]
 [0.45]]

Test Case 3 - Softmax activation, last layer
dA_prev: [[ 0.11085644 -0.07754183]
 [ 0.11085644 -0.06461819]
 [ 0.11085644 -0.03877091]]
dW: [[-3.05143585e-02  3.59706651e-02 -7.20960113e-05]
 [ 4.34379966e-02 -1.01233888e-02  3.23811914e-02]]
db: [[0.05528403]
 [0.00933416]]



In [53]:
def initialize_adam_parameters(W, b):
    
    vdW = np.zeros(W.shape)
    vdb = np.zeros(b.shape)

    sdW = np.zeros(W.shape)
    sdb = np.zeros(b.shape)

    return (vdW, vdb, sdW, sdb) 

In [59]:
def compute_first_momentum(dW, db, vdW, vdb, beta, t):
    
    vdW = (beta * vdW) + ((1 - beta) * dW)
    vdb = (beta * vdb) + ((1 - beta) * db)

    vdW = vdW / (1 - np.power(beta, t))
    vdb = vdb / (1 - np.power(beta, t))

    return (vdW, vdb)

In [60]:
def compute_second_momentum(dW, db, sdW, sdb, beta, t):
    
    sdW = (beta * sdW) + ((1 - beta) * np.power(dW, 2))
    sdb = (beta * sdb) + ((1 - beta) * np.power(db, 2))

          
    sdW = sdW / (1 - np.power(beta, t))
    sdb = sdb / (1 - np.power(beta, t))

    return (sdW, sdb)

In [61]:
def update_parameters(dW, db, W, b, learning_rate, beta1, beta2, epsilon, t, adam_params=None):

    if adam_params is None:

        (vdW, vdb, sdW, sdb) = initialize_adam_parameters(W=W, b=b)
        
    else:
        
        vdW = adam_params["vdW"]
        vdb = adam_params["vdb"]
        sdW = adam_params["sdW"]
        sdb = adam_params["sdb"]

        (vdW, vdb) = compute_first_momentum(dW=dW, db=db, vdW=vdW, vdb=vdb, beta=beta1, t=t)
        (sdW, sdb) = compute_second_momentum(dW=dW, db=db, sdW=sdW, sdb=sdb, beta=beta2, t=t)
               
    
    W -= learning_rate * (vdW / np.sqrt(sdW + epsilon))
    b -= learning_rate * (vdb / np.sqrt(sdb + epsilon))

    adam_params_ = {
        
        "vdW": vdW,
        "vdb": vdb,
        "sdW": sdW,
        "sdb": sdb
        
    }
    
    return (W, b, adam_params_)

In [63]:
# Initialize parameters
np.random.seed(1)
W = np.random.randn(3, 2)
b = np.random.randn(3, 1)

# Mock gradients
dW = np.random.randn(3, 2)
db = np.random.randn(3, 1)

# Adam parameters
learning_rate = 0.01
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
t = 1

# First update
print("Initial W:", W)
print("Initial b:", b)
(W, b, adam_params) = update_parameters(dW, db, W, b, learning_rate, beta1, beta2, epsilon, t)
print("Updated W after first update:", W)
print("Updated b after first update:", b)

# Second update (with t=2)
t = 2
dW = np.random.randn(3, 2)
db = np.random.randn(3, 1)
(W, b, adam_params) = update_parameters(dW, db, W, b, learning_rate, beta1, beta2, epsilon, t, adam_params)
print("Updated W after second update:", W)
print("Updated b after second update:", b)

Initial W: [[ 1.62434536 -0.61175641]
 [-0.52817175 -1.07296862]
 [ 0.86540763 -2.3015387 ]]
Initial b: [[ 1.74481176]
 [-0.7612069 ]
 [ 0.3190391 ]]
Updated W after first update: [[ 1.62434536 -0.61175641]
 [-0.52817175 -1.07296862]
 [ 0.86540763 -2.3015387 ]]
Updated b after first update: [[ 1.74481176]
 [-0.7612069 ]
 [ 0.3190391 ]]
Updated W after second update: [[ 1.61690404 -0.61919778]
 [-0.52073038 -1.08040999]
 [ 0.85796626 -2.30898006]]
Updated b after second update: [[ 1.7373704 ]
 [-0.75376553]
 [ 0.32648046]]


In [67]:
    def compute_cost(A, Y, loss):
      
        cost = 0

        if loss == "cross entropy":
            log_probs = np.multiply(-np.log(A + 1e-15), Y) + np.multiply(-np.log(1 - A + 1e-15), 1 - Y)
            cost = np.sum(log_probs)

        elif loss == "categorical cross entropy":
            A = np.clip(A, 1e-15, 1 - 1e-15)
            cost = -np.sum(Y * np.log(A))

        return cost

In [68]:
def random_mini_batches(X, Y, mini_batch_size, seed):
    
    np.random.seed(seed)
    m = X.shape[0]  
    num_classes = Y.shape[0] 

    permutation = np.random.permutation(m)
    shuffled_X = X[permutation]
    shuffled_Y = Y[:, permutation]

    num_complete_minibatches = m // mini_batch_size
    mini_batches = []

    for k in range(num_complete_minibatches):
        start_idx = k * mini_batch_size
        end_idx = (k + 1) * mini_batch_size
        mini_batch_X = shuffled_X[start_idx:end_idx]
        mini_batch_Y = shuffled_Y[:, start_idx:end_idx]
        mini_batches.append((mini_batch_X, mini_batch_Y))

    if m % mini_batch_size != 0:
        start_idx = num_complete_minibatches * mini_batch_size
        mini_batch_X = shuffled_X[start_idx:]
        mini_batch_Y = shuffled_Y[:, start_idx:]
        mini_batches.append((mini_batch_X, mini_batch_Y))

    return mini_batches